# 📉 Host Country Advantage Decay — Academic Paper Notebook
### The Beautiful Data · Signal & Structure by Krystie Dickson

**Research question:** Has the performance boost a World Cup host nation receives declined over the tournament's 96-year history?

**Method:** Two-stage Dynamic Linear Model estimated via Kalman filter
- Stage 1: Per-tournament cross-sectional OLS → β̂_t (raw host advantage)
- Stage 2: DLM/Kalman smoother on the β̂_t series → time-varying HA trajectory
- Validation: 2026 three-host natural experiment (USA, Mexico, Canada)

**Target journal:** Journal of Quantitative Analysis in Sports (JQAS)

**Key innovation:** First paper to apply a time-varying coefficient model to host-country advantage in major football tournaments.

---

## 1 · Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.statespace.structural import UnobservedComponents
from scipy import stats
import warnings, json
from pathlib import Path

try:
    from pykalman import KalmanFilter
    HAS_PYKALMAN = True
except ImportError:
    HAS_PYKALMAN = False
    print("Note: pykalman not found — using statsmodels UnobservedComponents")

warnings.filterwarnings('ignore')

ROOT      = Path('..')
PROCESSED = ROOT / 'data' / 'processed'
STATIC    = ROOT / 'data' / 'static'
OUT_DIR   = ROOT / 'outputs' / 'charts'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Colors
GREEN='#3CAC3B'; GOLD='#C9A227'; BLUE='#2A398D'
RED='#E61D25'; DARK='#12183A'; GRAY='#6B7280'; WHITE='#FFFFFF'

plt.rcParams.update({
    'figure.facecolor':WHITE,'axes.facecolor':WHITE,
    'axes.spines.top':False,'axes.spines.right':False,
    'axes.spines.left':False,'axes.grid':True,
    'grid.color':'#EEEEEC','grid.linewidth':0.8,
    'font.family':'sans-serif','font.size':11,
    'axes.titlesize':13,'axes.titleweight':'bold',
})
print("✓ Setup complete")
print(f"  pykalman available: {HAS_PYKALMAN}")


## 2 · Load data and construct outcome variable

In [ ]:
# Load historical match data
hist = pd.read_csv(STATIC / 'WorldCupMatches.csv')
wc   = pd.read_csv(STATIC / 'WorldCups.csv')

print(f"Matches: {len(hist)}")
print(f"Tournaments: {wc.shape}")
print(f"WC columns: {list(wc.columns)}")
print()
print(wc[['Year','Host']].head(10) if 'Host' in wc.columns else wc.head(5))


In [ ]:
# ── Tournament metadata ───────────────────────────────────────────────────
# Manual host country data (reliable; supplements Kaggle dataset)
HOSTS = {
    1930:'Uruguay',1934:'Italy',1938:'France',1950:'Brazil',
    1954:'Switzerland',1958:'Sweden',1962:'Chile',1966:'England',
    1970:'Mexico',1974:'West Germany',1978:'Argentina',1982:'Spain',
    1986:'Mexico',1990:'Italy',1994:'United States',1998:'France',
    2002:'South Korea',  # co-host; Japan also (2002 special case)
    2006:'Germany',2010:'South Africa',2014:'Brazil',
    2018:'Russia',2022:'Qatar',
    # 2026: three hosts
    2026:'United States',  # will test all three separately
}

# Tournament format — max rounds possible (for normalisation)
MAX_ROUNDS = {
    1930:3, 1934:4, 1938:4, 1950:3, 1954:4, 1958:4, 1962:4,
    1966:4, 1970:4, 1974:4, 1978:4, 1982:4, 1986:5, 1990:5,
    1994:5, 1998:5, 2002:5, 2006:5, 2010:5, 2014:5,
    2018:5, 2022:5, 2026:6,  # extra round in 2026 (R32 added)
}

print(f"Host countries: {len(HOSTS)} tournaments mapped")


In [ ]:
# ── Construct depth score (normalised performance) ───────────────────────
# Map stage names to rounds survived
ROUND_MAP = {
    'Group Stage': 0, 'First round': 0,
    'Second round': 1, 'Round of 16': 1,
    'Quarter-finals': 2, 'Quarterfinals': 2,
    'Semi-finals': 3, 'Semifinals': 3,
    'Third place': 4, 'Play-off for third place': 4,
    'Final': 4,
}

def stage_to_round(stage_str):
    for key, val in ROUND_MAP.items():
        if key.lower() in str(stage_str).lower():
            return val
    return 0

# For each team-tournament, find deepest round reached
hist['round_reached'] = hist['Stage'].apply(stage_to_round)

# Add 1 for teams that reached the Final (they survived that round too)
finals = hist[hist['Stage'].str.contains('Final', na=False)]
final_teams = set()
for _, row in finals.iterrows():
    final_teams.add((row['Year'], row['Home Team Name'].strip()))
    final_teams.add((row['Year'], row['Away Team Name'].strip()))

# Find max round per team per year
team_year_max = hist.groupby(['Year','Home Team Name'])['round_reached'].max().reset_index()
team_year_max.columns = ['year','team','max_round']
awaymax = hist.groupby(['Year','Away Team Name'])['round_reached'].max().reset_index()
awaymax.columns = ['year','team','max_round']
team_year_max = pd.concat([team_year_max, awaymax]).groupby(
    ['year','team'])['max_round'].max().reset_index()

# Depth score = rounds_survived / max_rounds_in_tournament
team_year_max['max_possible'] = team_year_max.year.map(MAX_ROUNDS)
team_year_max['depth_score'] = team_year_max.max_round / team_year_max.max_possible
team_year_max['team'] = team_year_max.team.str.strip()

print(f"✓ Depth scores computed: {len(team_year_max)} team-tournament records")
print(f"  Score range: {team_year_max.depth_score.min():.2f} to {team_year_max.depth_score.max():.2f}")
print()
# Show host countries' depth scores
host_perf = []
for year, host in HOSTS.items():
    row = team_year_max[(team_year_max.year==year) & 
                        (team_year_max.team.str.contains(host.split()[0], na=False))]
    if len(row) > 0:
        host_perf.append({'year':year,'host':host,'depth_score':row.iloc[0].depth_score,
                          'max_round':row.iloc[0].max_round})

host_df = pd.DataFrame(host_perf).set_index('year')
print("Host country performance (depth score 0-1):")
print(host_df.to_string())


## 3 · Simplified Elo proxy for team strength

In [ ]:
# Approximate historical Elo from win rate at each tournament
# (Proper analysis: download full Elo series from eloratings.net)
# For now: use cumulative win rate as a strength proxy

team_wins = hist.groupby(['Year','Home Team Name']).apply(
    lambda df: (df['Home Team Goals'] > df['Away Team Goals']).sum()
).reset_index(name='home_wins')
team_games = hist.groupby(['Year','Home Team Name']).size().reset_index(name='games')

# Simple elo proxy: historical win% up to that year
all_wins = {}
for _, row in hist.iterrows():
    y, ht, at = row['Year'], row['Home Team Name'].strip(), row['Away Team Name'].strip()
    hg, ag = row['Home Team Goals'], row['Away Team Goals']
    if pd.isna(hg) or pd.isna(ag): continue
    for t in [ht, at]:
        if t not in all_wins: all_wins[t] = {'w':0,'g':0}
    all_wins[ht]['g'] += 1; all_wins[at]['g'] += 1
    if hg > ag: all_wins[ht]['w'] += 1
    elif ag > hg: all_wins[at]['w'] += 1

elo_proxy = {t: v['w']/max(v['g'],1) for t,v in all_wins.items()}

team_year_max['elo_proxy'] = team_year_max.team.map(elo_proxy).fillna(0.3)
team_year_max['is_host'] = team_year_max.apply(
    lambda r: 1 if r.team == HOSTS.get(r.year,'') or
                   HOSTS.get(r.year,'').startswith(r.team.split()[0])
    else 0, axis=1)

print(f"Host flag set for {team_year_max.is_host.sum()} team-tournament rows")
print(f"Elo proxy range: {elo_proxy and min(elo_proxy.values()):.2f} — {elo_proxy and max(elo_proxy.values()):.2f}")


## 4 · Stage 1 — estimate raw host advantage per tournament

In [ ]:
# For each tournament: depth_score ~ is_host + elo_proxy
# β̂_t = host advantage coefficient at tournament t

stage1_results = []

for year in sorted(team_year_max.year.unique()):
    df_t = team_year_max[team_year_max.year == year].copy()
    df_t = df_t.dropna(subset=['depth_score','elo_proxy'])
    
    if len(df_t) < 5 or df_t.is_host.sum() == 0:
        continue
    
    try:
        X = sm.add_constant(df_t[['is_host','elo_proxy']])
        y = df_t['depth_score']
        m = sm.OLS(y, X).fit()
        
        stage1_results.append({
            'year':       year,
            'beta_ha':    m.params.get('is_host', np.nan),
            'se_ha':      m.bse.get('is_host', np.nan),
            'n_teams':    len(df_t),
            'host':       HOSTS.get(year,'Unknown'),
            'host_score': df_t[df_t.is_host==1]['depth_score'].mean(),
            'mean_score': df_t['depth_score'].mean(),
            'r2':         m.rsquared,
        })
    except Exception as e:
        pass

s1 = pd.DataFrame(stage1_results).set_index('year')
s1['weight'] = 1 / s1.se_ha.clip(lower=0.01)**2  # inverse-variance weight

print("Stage 1 results — raw host advantage β̂_t by tournament:")
print(s1[['host','beta_ha','se_ha','n_teams','host_score']].to_string())


## 5 · Stage 2 — Dynamic Linear Model via Kalman filter

In [ ]:
# ── Kalman filter / Local level DLM ──────────────────────────────────────
# Observation:  β̂_t = β_t + v_t,  v_t ~ N(0, σ²_v) = se_ha²  (known)
# State:        β_t = β_{t-1} + η_t, η_t ~ N(0, σ²_η)  (to estimate)
#
# H₀: σ²_η = 0  (constant HA — no decay)
# H₁: σ²_η > 0  (time-varying HA — decay present)

ha_series = s1['beta_ha'].dropna()
se_series = s1['se_ha'].dropna().reindex(ha_series.index).fillna(ha_series.std())
years_s2  = ha_series.index.values

print(f"Stage 2 input: {len(ha_series)} tournament-level HA estimates")
print(f"HA range: {ha_series.min():.3f} to {ha_series.max():.3f}")
print(f"Mean HA: {ha_series.mean():.3f} (raw average across all tournaments)")

# ── Method A: statsmodels UnobservedComponents (Local Level) ──────────────
mod = UnobservedComponents(ha_series.values, 'local level',
                           obs_cov=np.diag(se_series.values**2))
try:
    res_dlm = mod.fit(disp=False, method='nm')
    
    # Extract smoothed state (time-varying HA estimate)
    smoothed  = res_dlm.smoothed_state[0]
    smoothed_se = np.sqrt(res_dlm.smoothed_state_cov[0,0])
    
    sigma_eta = np.sqrt(res_dlm.params[0])
    sigma_v   = se_series.mean()
    snr = sigma_eta / sigma_v
    
    print(f"\n✓ DLM fitted successfully")
    print(f"  σ_η (state noise SD):  {sigma_eta:.4f}")
    print(f"  Signal-to-noise ratio: {snr:.3f}")
    print(f"  (SNR > 0.1 suggests meaningful time variation)")
    print(f"  Log-likelihood: {res_dlm.llf:.2f}")
    print(f"  AIC: {res_dlm.aic:.2f}")
    
    DLM_FIT = True
except Exception as e:
    print(f"DLM fitting error: {e}")
    DLM_FIT = False


In [ ]:
# ── Test H₀: constant HA (σ²_η = 0) ─────────────────────────────────────
if DLM_FIT:
    # Restricted model (constant HA)
    mod_restricted = sm.OLS(ha_series.values, 
                            sm.add_constant(np.ones(len(ha_series)))).fit()
    
    # Simple LR test approximation
    lr_stat  = 2 * (res_dlm.llf - mod_restricted.llf)
    p_value  = stats.chi2.sf(lr_stat, df=1)
    
    print("\nLikelihood Ratio Test: H₀: σ²_η = 0 (constant HA)")
    print(f"  LR statistic: {lr_stat:.3f}")
    print(f"  p-value:      {p_value:.4f}")
    print(f"  Decision:     {'Reject H₀ — HA is time-varying' if p_value<0.05 else 'Fail to reject H₀'}")
    if p_value < 0.1:
        print(f"\n  → Evidence that host advantage HAS changed over time")
    else:
        print(f"\n  → Insufficient evidence of time-variation (small sample T={len(ha_series)})")


## 6 · The Kalman smoother — visualising the decay

In [ ]:
if DLM_FIT:
    fig, axes = plt.subplots(2, 1, figsize=(14, 10))
    fig.suptitle("Host Country Advantage at the World Cup — A Time-Varying Analysis",
                 fontsize=15, fontweight='bold', color=DARK)
    
    # ── Chart 1: Raw HA with Kalman smoothed trend ────────────────────────
    ax = axes[0]
    
    # Raw estimates with error bars
    ax.errorbar(years_s2, ha_series.values, yerr=2*se_series.values,
                fmt='o', color=BLUE, ecolor='#AAAACC', elinewidth=1.5,
                capsize=4, markersize=6, markerfacecolor=WHITE,
                markeredgewidth=2, zorder=3, label='Stage 1 estimate (±2 SE)')
    
    # Kalman smoothed trend
    ax.plot(years_s2, smoothed, color=RED, linewidth=2.5, zorder=4,
            label='Kalman smoothed HA trajectory')
    
    # Confidence band
    ax.fill_between(years_s2,
                    smoothed - 1.96*smoothed_se,
                    smoothed + 1.96*smoothed_se,
                    color=RED, alpha=0.12, label='95% credible band')
    
    # Zero line
    ax.axhline(0, color=DARK, linewidth=1, linestyle='-', alpha=0.4)
    ax.text(1932, 0.02, 'No host advantage', fontsize=8, color=GRAY)
    
    # Annotate notable tournaments
    for year, host in [(1966,'England'),(1974,'W.Germany'),(1994,'USA'),
                       (1998,'France'),(2014,'Brazil')]:
        if year in s1.index:
            ax.annotate(host, xy=(year, s1.loc[year,'beta_ha']),
                        xytext=(year+0.5, s1.loc[year,'beta_ha']+0.08),
                        fontsize=7, color=DARK, alpha=0.7)
    
    ax.set_ylabel("Host advantage coefficient (β̂_t)")
    ax.set_xlabel("Tournament year")
    ax.legend(fontsize=9, frameon=False, loc='upper right')
    ax.set_title("Raw host advantage estimates + Kalman smoothed trajectory")
    
    # ── Chart 2: Rolling window comparison ───────────────────────────────
    ax = axes[1]
    
    # 5-tournament rolling window
    roll_mean = ha_series.rolling(5, center=True, min_periods=3).mean()
    roll_std  = ha_series.rolling(5, center=True, min_periods=3).std()
    
    ax.fill_between(years_s2, 
                    roll_mean - roll_std, roll_mean + roll_std,
                    color=BLUE, alpha=0.15, label='Rolling ±1 SD')
    ax.plot(years_s2, roll_mean, color=BLUE, linewidth=2,
            label='5-tournament rolling mean', zorder=3)
    ax.plot(years_s2, smoothed, color=RED, linewidth=2,
            linestyle='--', label='Kalman smoother', zorder=4)
    ax.axhline(0, color=DARK, linewidth=1, alpha=0.4)
    ax.axvline(2006, color=GOLD, linewidth=2, alpha=0.6, linestyle=':')
    ax.text(2006.5, ax.get_ylim()[1]*0.8 if ax.get_ylim()[1]>0 else 0.3,
            '2006', fontsize=8, color=GOLD)
    
    ax.set_ylabel("Host advantage")
    ax.set_xlabel("Tournament year")
    ax.legend(fontsize=9, frameon=False)
    ax.set_title("Rolling window vs Kalman smoother — do they agree?")
    
    plt.tight_layout()
    cp = OUT_DIR / 'host_advantage_kalman.png'
    plt.savefig(cp, dpi=150, bbox_inches='tight', facecolor=WHITE)
    plt.show()
    print(f"✓ Kalman chart saved to {cp}")
    print("  This is your Figure 2 for the JQAS paper")


## 7 · 2026 out-of-sample validation

In [ ]:
# ── Forecast HA for 2026 hosts ───────────────────────────────────────────
if DLM_FIT:
    # One-step ahead forecast for 2026 (first tournament after sample)
    forecast_mean = smoothed[-1]   # last smoothed state
    forecast_var  = smoothed_se**2 + res_dlm.params[0]  # state var + noise
    forecast_se   = np.sqrt(forecast_var)
    
    print("="*55)
    print("2026 OUT-OF-SAMPLE FORECAST")
    print("="*55)
    print(f"Predicted host advantage for 2026: {forecast_mean:+.3f}")
    print(f"95% prediction interval:  [{forecast_mean-1.96*forecast_se:.3f}, "
          f"{forecast_mean+1.96*forecast_se:.3f}]")
    print()
    
    # Hosts in 2026: USA (Elo ~1808), Mexico (~1810), Canada (~1782)
    hosts_2026 = [
        {'country':'United States','elo_proxy':0.52,'previous_host':True},
        {'country':'Mexico',       'elo_proxy':0.53,'previous_host':True},
        {'country':'Canada',       'elo_proxy':0.48,'previous_host':False},
    ]
    
    print("Predicted performance with HA:")
    for h in hosts_2026:
        base_expected = h['elo_proxy'] * 0.5   # simplified
        ha_adjusted   = base_expected + forecast_mean
        print(f"  {h['country']:<20} Expected depth score: {ha_adjusted:.2f}  "
              f"({'prior host' if h['previous_host'] else 'first time'})")

# ── Load 2026 actual performance if available ────────────────────────────
matches_path = PROCESSED / 'latest_matches.json'
if matches_path.exists():
    live = pd.read_json(matches_path)
    finished = live[live.status=='FINISHED']
    
    hosts_2026_teams = ['United States','Canada','Mexico']
    for host in hosts_2026_teams:
        host_games = finished[
            (finished.home_team.str.contains(host.split()[0], na=False)) |
            (finished.away_team.str.contains(host.split()[0], na=False))
        ]
        if len(host_games) > 0:
            wins  = sum(
                (r.home_score > r.away_score and host.split()[0] in r.home_team) or
                (r.away_score > r.home_score and host.split()[0] in r.away_team)
                for _, r in host_games.iterrows() if pd.notna(r.home_score)
            )
            print(f"  {host}: {len(host_games)} games played, {wins} wins (2026 actual)")


In [ ]:
# ── Robustness: Bai-Perron structural break test (simplified) ────────────
if DLM_FIT and len(ha_series) >= 10:
    print("="*55)
    print("ROBUSTNESS: Structural break in HA series")
    print("="*55)
    
    # Test for a single break at each possible year
    best_f   = 0
    best_bp  = None
    n = len(ha_series)
    
    for bp in range(3, n-3):  # require at least 3 obs each side
        y1 = ha_series.values[:bp]
        y2 = ha_series.values[bp:]
        
        rss_total = np.sum((ha_series.values - ha_series.mean())**2)
        rss_1     = np.sum((y1 - y1.mean())**2)
        rss_2     = np.sum((y2 - y2.mean())**2)
        rss_bp    = rss_1 + rss_2
        
        f_stat = ((rss_total - rss_bp) / 1) / (rss_bp / (n - 2))
        if f_stat > best_f:
            best_f  = f_stat
            best_bp = bp
    
    break_year = years_s2[best_bp] if best_bp else None
    print(f"Most likely structural break: {break_year}")
    print(f"F-statistic: {best_f:.3f}")
    print(f"Pre-break mean HA:  {ha_series.values[:best_bp].mean():+.3f}")
    print(f"Post-break mean HA: {ha_series.values[best_bp:].mean():+.3f}")
    print()
    print("Interpretation:")
    pre  = ha_series.values[:best_bp].mean()
    post = ha_series.values[best_bp:].mean()
    direction = "declined" if post < pre else "increased"
    print(f"  Host advantage {direction} after {break_year}")
    print(f"  Change: {post-pre:+.3f} depth score units")


## 8 · Paper-ready output summary

**Run this cell to generate the key numbers for your paper.**

In [ ]:
print("="*60)
print("HOST COUNTRY ADVANTAGE DECAY PAPER — KEY RESULTS")
print("For JQAS submission")
print("="*60)
print()
print(f"Sample: {len(ha_series)} World Cup tournaments ({years_s2[0]}–{years_s2[-1]})")
print()
print(f"Mean host advantage (full sample): {ha_series.mean():+.3f} depth score units")
print(f"  = host nations finish {ha_series.mean()/0.2*100:.0f}% deeper in the tournament")
print(f"    than a team of equal Elo strength")
print()
if DLM_FIT:
    print(f"DLM results:")
    print(f"  σ_η (state SD):        {sigma_eta:.4f}")
    print(f"  Signal-to-noise ratio: {snr:.3f}")
    print(f"  Most recent HA est.:   {smoothed[-1]:+.3f} ({years_s2[-1]})")
    print(f"  2026 forecast:         {forecast_mean:+.3f} "
          f"[{forecast_mean-1.96*forecast_se:.3f}, {forecast_mean+1.96*forecast_se:.3f}]")
    print()
    print(f"Structural break:")
    print(f"  Most likely at {break_year} (F={best_f:.2f})")
    print(f"  Pre-{break_year} mean HA:  {ha_series.values[:best_bp].mean():+.3f}")
    print(f"  Post-{break_year} mean HA: {ha_series.values[best_bp:].mean():+.3f}")
print()
print("Charts saved:")
for f in ['host_advantage_kalman.png']:
    p = OUT_DIR / f
    print(f"  {'✓' if p.exists() else '○'} {f}")
print()
print("Next steps:")
print("  1. Download Elo ratings from eloratings.net → replace elo_proxy")
print("  2. Add 2026 results as they come in → update 2026 validation section")
print("  3. Run notebook again at tournament end for final validation")
print("  4. Draft manuscript: Introduction → Literature → Methods → Results → Discussion")
